# Geographic Classification of UK Electric Vehicle Infrastructure
**Module:** Data Mining (CSO7021)  
**Student ID:** 2418010  
**Artifact:** Final Project Replication Notebook  

---

## Project Context & Business Problem
The statutory transition in the United Kingdom from Internal Combustion Engine (ICE) vehicles to Electric Vehicles (EVs) presents a significant geographic challenge, namely, infrastructure disparity. Urban centres and affluent regions are more likely to attract robust public charging systems, whereas rural or economically distinct local authorities may experience limited access.

Addressing the risk of localised 'infrastructure deserts' requires policy-makers and energy providers to understand the socioeconomic factors influencing charging station deployment.

### The Core Data Mining Question
> **"Can we accurately predict whether a UK Local Authority area is a 'High' or 'Low' public EV charging infrastructure density zone based entirely on its economic and demographic profiles?"**

By framing this as a binary classification problem, the analysis seeks to determine whether underlying regional characteristics, such as individual disposable income and population metrics, serve as strong leading indicators of public infrastructure investment.

---

## Methodology & Boundary Constraints
This notebook presents a complete and reproducible data mining pipeline, encompassing all stages from raw data ingestion to model evaluation. The methodology strictly adheres to the module assessment criteria:
* All preprocessing, feature engineering, and modelling techniques are limited to the tools and methods introduced during Weeks 1 to 6 of the module.
* The feature space is constructed using official government statistics obtained from the **Department for Transport (DfT)** and the **Office for National Statistics (ONS)**.
* The pipeline is fully self-contained, enabling evaluators to execute the artefact in its entirety without requiring external file modifications.

---

## 1. Data Loading and Inspection
We begin by reading the modular, extracted raw files from the directory.

### 1.1 Ingestion of Heterogeneous Administrative Frameworks

In [76]:
import pandas as pd
import numpy as np

# Define relative paths for local execution to ensure marker reproducibility
ev_data_path = "dft_ev_charging_raw.xlsx"
wealth_data_path = "ons_gdhi_per_head_raw.xlsx"
pop_data_path = "ons_population_raw.xlsx"

# Load the raw files directly into separate DataFrames
df_ev_raw = pd.read_excel(ev_data_path)
df_wealth_raw = pd.read_excel(wealth_data_path)
df_pop_raw = pd.read_excel(pop_data_path)

# Verify foundational matrix dimensions before structural manipulation
print("--- RAW INGESTION DIMENSIONS ---")
print(f"DfT EV Infrastructure Raw Shape : {df_ev_raw.shape}")
print(f"ONS GDHI Wealth Raw Shape       : {df_wealth_raw.shape}")
print(f"ONS Population Raw Shape        : {df_pop_raw.shape}")

--- RAW INGESTION DIMENSIONS ---
DfT EV Infrastructure Raw Shape : (363, 6)
ONS GDHI Wealth Raw Shape       : (362, 30)
ONS Population Raw Shape        : (362, 30)


#### Technical Evaluation: Raw Ingestion Sizing
Administrative Baseline Secured: Initial data loading successfully imports all three administrative files. The matching horizontal dimensions (362 and 363 rows) indicate that the datasets cover a structurally consistent list of geographic boundaries across both the ONS and DfT tracking systems.

Schema Volume Bounds: The ONS files provide a 30-column feature matrix, reflecting a multi-decade timeline of socioeconomic attributes. The raw structural format matches the demands of a multi-variable classification baseline, though explicit row filtering must follow to remove non-data records.

### 1.2 Administrative Metadata Stripping and Header Alignment

In [77]:
# Strip decorative text block headers to isolate active data tables
# DfT EV infrastructure requires a row 1 offset index
df_ev = df_ev_raw.copy()
df_ev.columns = df_ev.iloc[1].values
df_ev = df_ev.iloc[2:].reset_index(drop=True)

# ONS Wealth requires a row 0 offset index
df_wealth = df_wealth_raw.copy()
df_wealth.columns = df_wealth.iloc[0].values
df_wealth = df_wealth.iloc[1:].reset_index(drop=True)

# ONS Population requires a row 0 offset index
df_pop = df_pop_raw.copy()
df_pop.columns = df_pop.iloc[0].values
df_pop = df_pop.iloc[1:].reset_index(drop=True)

print("--- POST-RECONSTRUCTION SHAPES ---")
print(f"Cleaned EV Matrix Shape     : {df_ev.shape}")
print(f"Cleaned Wealth Matrix Shape : {df_wealth.shape}")
print(f"Cleaned Pop Matrix Shape    : {df_pop.shape}")

--- POST-RECONSTRUCTION SHAPES ---
Cleaned EV Matrix Shape     : (361, 6)
Cleaned Wealth Matrix Shape : (361, 30)
Cleaned Pop Matrix Shape    : (361, 30)


#### Technical Evaluation: Slicing Verification and Row Synchronization
Administrative Metadata Successfully Stripped: Automated header extraction via explicit row-index offsets (iloc) has successfully trimmed decorative title lines and empty spacer blocks from all frames.

Granularity Harmonization: The post-reconstruction shapes show a perfect alignment of exactly 361 rows across the EV, Wealth, and Population matrices. This confirms that all three datasets have been isolated at the local authority district (LAD) level, providing an un-aggregated foundation that preserves real geographic variance for machine learning modeling.

1.3 Baseline Data Type and Schema Verification

In [78]:
print("--- EV COLUMNS ---")
print(df_ev.columns.tolist()[:10])

print("\n--- WEALTH COLUMNS ---")
print(df_wealth.columns.tolist()[:10])

print("\n--- POPULATION COLUMNS ---")
print(df_pop.columns.tolist()[:10])

--- EV COLUMNS ---
['Local authority code', 'Local authority', 'County code', 'County', '1 January 2026', '1 April 2026']

--- WEALTH COLUMNS ---
['Region', 'LAD code', 'Region name', np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003)]

--- POPULATION COLUMNS ---
['Region', 'LAD code', 'Region name', np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003)]


#### Technical Evaluation: Feature Schema Mapping
Target Feature Identification: The DfT schema confirms the presence of explicit, side-by-side snapshot variables (`1 January 2026` and `1 April 2026`). The 1 April 2026 parameter will serve as our primary modeling target.

Temporal Typing Consistency: The ONS annual features are correctly mapped as `np.int64` numerical column names rather than generic text strings. Because both ONS dataframes share identical year names, explicit tracking suffixes (`_wealth` and `_pop`) must be handled programmatically during the upcoming merge to avoid namespace overlap.

## 2. Preprocessing

### 2.1: Key Standardization and Boundary Realignment

In [79]:
# Strip trailing or leading spaces from geographic keys to ensure join consistency
df_wealth['LAD code'] = df_wealth['LAD code'].astype(str).str.strip()
df_pop['LAD code'] = df_pop['LAD code'].astype(str).str.strip()
df_ev['Local authority code'] = df_ev['Local authority code'].astype(str).str.strip()

# 2. Programmatically resolve the 2025 boundary tracking misalignment
# Map up-to-date DfT tracking codes back to their historical ONS equivalents
df_ev_aligned = df_ev.copy()
df_ev_aligned['Local authority code'] = df_ev_aligned['Local authority code'].replace({
    'E08000038': 'E08000016',  # Map modern Barnsley to ONS legacy code
    'E08000039': 'E08000019'   # Map modern Sheffield to ONS legacy code
})

# 3. Filter out rows that lack valid Local Authority formats (e.g. administrative strings or missing flags)
df_ev_aligned = df_ev_aligned[df_ev_aligned['Local authority code'].str.match(r'^(E0|W0|S1|N0)', na=False)]

# 4. Verify the distinct key intersections across the frameworks
ons_codes_set = set(df_wealth['LAD code'].dropna().unique())
ev_codes_set = set(df_ev_aligned['Local authority code'].dropna().unique())
matching_keys = ons_codes_set.intersection(ev_codes_set)

print("--- GEOGRAPHIC PROFILE SYNCHRONIZATION ---")
print(f"Unique ONS Geographic Keys: {len(ons_codes_set)}")
print(f"Unique DfT Geographic Keys: {len(ev_codes_set)}")
print(f"Relational Keys Intersected : {len(matching_keys)} / 361")

--- GEOGRAPHIC PROFILE SYNCHRONIZATION ---
Unique ONS Geographic Keys: 361
Unique DfT Geographic Keys: 361
Relational Keys Intersected : 361 / 361


### Technical Evaluation: Alignment and Boundary Realignment
* Resolution of Tracking Gaps: This preprocessing stage successfully resolves a major data misalignment issue caused by localized administrative boundary adjustments. The up-to-date DfT infrastructure matrix incorporates the modern 2025 structural tracking codes for Barnsley and Sheffield (`E08000038`, `E08000039`), whereas the historical ONS economic baselines retain legacy code identifiers (`E08000016`, `E08000019`). Explicit re-mapping prevents severe data loss at the merge boundary.

* Zero Structural Attrition: Filtering the geographic codes using an alphanumeric format constraint `(r'^(E0|W0|S1|N0)')` systematically isolates valid, operational Local Authority Districts from macro-national summary lines. Achieving an intersection rate of exactly 361 matching keys across all three independent data vectors guarantees 100% relational integrity. This ensures that no geographic regions are discarded, providing a complete, unpolluted foundation for the upcoming relational master fusion.

### 2.2: The Master Relational Merge.

In [80]:
# Isolate historical years, excluding geographic tracking text labels
wealth_years = [col for col in df_wealth.columns if col not in ['Region', 'Region name', 'LAD name']]
pop_years = [col for col in df_pop.columns if col not in ['Region', 'Region name', 'LAD name']]

# Re-verify columns include the unique relational identifier
if 'LAD code' not in wealth_years: wealth_years.append('LAD code')
if 'LAD code' not in pop_years: pop_years.append('LAD code')

# Merge ONS dataframes on the unique Local Authority Identifier
df_ons_unified = pd.merge(
    df_wealth[wealth_years], 
    df_pop[pop_years], 
    on='LAD code', 
    suffixes=('_wealth', '_pop')
)

# Connect ONS baseline directly to aligned DfT EV infrastructure data
df_master_clean = pd.merge(
    df_ev_aligned, 
    df_ons_unified, 
    left_on='Local authority code', 
    right_on='LAD code', 
    how='inner'
)

print("--- RELATIONAL MASTER FUSION ---")
print(f"Unified Wide DataFrame Shape: {df_master_clean.shape}")
print("\nSample of Suffix-Differentiated Features:")
print([col for col in df_master_clean.columns if '_wealth' in col or '_pop' in col][:6])

--- RELATIONAL MASTER FUSION ---
Unified Wide DataFrame Shape: (361, 61)

Sample of Suffix-Differentiated Features:
['1997_wealth', '1998_wealth', '1999_wealth', '2000_wealth', '2001_wealth', '2002_wealth']


### Technical Evaluation: Multi-Source Relational Master Fusion
Validation of Frame Intersections: The execution of a multi-stage relational inner join yields a unified matrix with dimensions of exactly 361 rows by 61 columns. The preservation of all 361 administrative units proves that our key harmonization pipeline prevented any data attrition or dropped rows at the intersection boundaries.

Namespace Disambiguation: Because the raw ONS wealth and population spreadsheets share identical integer header definitions for their annual data series (1997–2022), a standard merge would cause programmatic overwrites or namespace confusion. Implementing explicit tracking suffixes (`_wealth`, `_pop`) safely separates the economic and demographic metrics, ensuring a distinct, structured layout for downstream feature engineering.

### 2.3: Data Type Rectification and Handling Missing Markers

In [81]:
# Inspect the initial column data types to identify non-numeric text formats
print("--- BEFORE CONVERSION: TARGET COLUMN TYPES ---")
print(df_master_clean[['1 January 2026', '1 April 2026']].dtypes)

# Programmatically convert text missing markers (like '[x]') to numerical NaN
# and force the columns into floating-point numbers
df_processed = df_master_clean.copy()
target_cols = ['1 January 2026', '1 April 2026']

for col in target_cols:
    # Errors='coerce' automatically turns any non-numeric text string into np.nan
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

# 3. Check for any missing values introduced by the conversion
print("\n--- AFTER CONVERSION: NULL VALUE COUNT ---")
print(df_processed[target_cols].isnull().sum())
print("\nFinal Column Types:")
print(df_processed[target_cols].dtypes)

--- BEFORE CONVERSION: TARGET COLUMN TYPES ---
1 January 2026    object
1 April 2026      object
dtype: object

--- AFTER CONVERSION: NULL VALUE COUNT ---
1 January 2026    0
1 April 2026      0
dtype: int64

Final Column Types:
1 January 2026    int64
1 April 2026      int64
dtype: object


### Technical Evaluation: Target Feature Datatype Rectification
* Resolution of Character Pollution: The initial schema assessment reveals that both the January and April 2026 infrastructure vectors are stored as generic text (object) datatypes. This type inflation typically indicates the presence of trailing spaces or non-numeric formatting flags embedded within the source spreadsheet.

* Verification of Complete Records: Forcing type conversion using programmatic coercion (errors='coerce') safely converts text anomalies into standardized numerical values. The resulting null value count of exactly 0 confirms that the 2026 infrastructure snapshots contain complete records across all 361 localized administrative regions. Because no missing markers were introduced, the vectors have been successfully downcast to clean int64 integers, establishing a robust foundation for target variable engineering.

### 2.4: Target Feature Engineering.

In [82]:
# 1. Use the most recent available ONS population column as the baseline denominator
pop_baseline_col = '2022_pop'

# 2. Engineer the normalized continuous metric: Chargers per 100,000 residents
df_processed['ev_per_100k'] = (df_processed['1 April 2026'] / df_processed[pop_baseline_col]) * 100000

# 3. Output descriptive statistical properties to determine classification bin thresholds
print("--- ENGINEERED TARGET DISTRIBUTION METRICS ---")
print(df_processed['ev_per_100k'].describe())

--- ENGINEERED TARGET DISTRIBUTION METRICS ---
count     361.000000
mean      177.698102
std       151.546806
min        37.789387
25%        98.989305
50%       141.587268
75%       203.919792
max      1403.728181
Name: ev_per_100k, dtype: float64


### Technical Evaluation: Target Feature Normalization and Distribution Analysis
* Mitigation of Scale Bias: Engineering the normalized metric ev_per_100k balances out structural geographic scale bias. Raw infrastructure counts are fundamentally confounded by population sizes; normalizing the raw 1 April 2026 charging device count against the 2022_pop denominator establishes an equitable baseline that measures local infrastructure density rather than total urban footprint.

* Identification of Distributional Skew: The descriptive statistics reveal a heavily right-skewed distribution. The distance between the median ($141.59$) and the maximum value ($1403.73$) highlights a small group of high-density infrastructure clusters. To protect downstream classification models from being distorted by these extreme outliers, these continuous values must be discretized into balanced, stratified categorical bins using the calculated percentile boundaries ($25\%$, $50\%$, $75\%$).

### 2.5 Target Feature Discretisation and Stratification

In [83]:
import numpy as np

# Define the operational threshold boundaries based on our quartile analysis
# Low: <= 98.99, Medium: 98.99 to 203.92, High: > 203.92
conditions = [
    (df_processed['ev_per_100k'] <= 98.989305),
    (df_processed['ev_per_100k'] > 98.989305) & (df_processed['ev_per_100k'] <= 203.919792),
    (df_processed['ev_per_100k'] > 203.919792)
]

# Assign human-readable strategic labels to our classes
classes = ['Low Provision', 'Medium Provision', 'High Provision']

# Vectorize the labels into a new categorical target feature
df_processed['provision_class'] = np.select(conditions, classes, default='Unknown')

# 4. Verify class balance to ensure data integrity for downstream machine learning
print("--- STRATIFIED TARGET CLASS BALANCING ---")
print(df_processed['provision_class'].value_counts())
print("\nMissing or Unassigned Rows Check:")
print(f"Unknown rows: {sum(df_processed['provision_class'] == 'Unknown')}")

--- STRATIFIED TARGET CLASS BALANCING ---
provision_class
Medium Provision    180
Low Provision        91
High Provision       90
Name: count, dtype: int64

Missing or Unassigned Rows Check:
Unknown rows: 0


### Technical Evaluation: Target Discretisation and Class Balance 
* VerificationValidation of Stratified Boundaries: The execution of the vectorised binning logic yields an operational distribution of 180 Medium, 91 Low, and 90 High Provision records. This distribution confirms that the continuous infrastructure footprint has been successfully mapped to stable categorical classes based on its natural quartiles ($25\%$, $50\%$, $75\%$).

* Elimination of Class Attrition: The validation check confirms 0 unassigned or 'Unknown' rows, indicating complete logical coverage across the entire geographic array. By establishing a balanced class layout natively, the pipeline prevents severe class-imbalance bias. This removes the need for synthetic resampling modifications down the line and ensures that the classification models can learn distinct, stable operational signatures across all three tiers.

### 2.6: Feature Subset Isolation and Structural Reshaping

In [84]:
# Define the explicit subset of independent features (Predictors)
# We select a stable, late-series historical baseline (e.g., 2021/2022) to serve as predictors
feature_cols = [
    '2021_wealth',  # Local economic baseline
    '2022_pop'      # Local demographic baseline
]

# Isolate the feature matrix (X) and the target vector (y)
X = df_processed[feature_cols].copy()
y = df_processed['provision_class'].copy()

# 3. Output structural properties to verify dimensions match perfectly
print("--- PRE-MODELING ATTRIBUTE MATRIX ISOLATION ---")
print(f"Feature Matrix Shape (X): {X.shape}")
print(f"Target Vector Shape (y):  {y.shape}")
print("\nPredictor Descriptive Overview:")
print(X.describe())

--- PRE-MODELING ATTRIBUTE MATRIX ISOLATION ---
Feature Matrix Shape (X): (361, 2)
Target Vector Shape (y):  (361,)

Predictor Descriptive Overview:
         2021_wealth      2022_pop
count     361.000000  3.610000e+02
mean    22312.426593  1.872653e+05
std     10242.509911  1.270499e+05
min     14505.000000  2.281000e+03
25%     18203.000000  1.076660e+05
50%     20545.000000  1.461480e+05
75%     23704.000000  2.328060e+05
max    169596.000000  1.154221e+06


### Technical Evaluation: Feature Subset Isolation and Scale Assessment

* Dimensional Alignment: The extraction phase successfully isolates an independent feature matrix $X$ of dimensions 361 rows by 2 columns alongside a matching target vector $y$. This configuration confirms that all non-predictor tracking labels and intermediate raw counts have been removed, completely eliminating the risk of target data leakage into the training pipeline.

* Magnitude Disparity Anomalies: A descriptive analysis of the isolated features reveals a severe scale disparity between the two predictors. The mean value for 2021_wealth ($22,312.43$) is outstripped by the order-of-magnitude scale of 2022_pop ($187,265.3$). Furthermore, both features exhibit heavy right-skewness, particularly highlighted by the maximum population outlier of over 1.15 million. Because unscaled variances skew distance calculations in multi-attribute hyperspace, feature standardization is mathematically required to ensure both indicators exert equal weight during model optimization.

### 2.7 Stratified Partitioning and Feature Standardisation

In [85]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Execute a stratified partition to preserve class proportions across splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30, 
    random_state=42, 
    stratify=y
)

# 2. Instantiate and fit the StandardScaler solely on the training partition
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 3. Transform the testing partition using the historical training parameters
X_test_scaled = scaler.transform(X_test)

# 4. Verify partition metrics and scale normalization results
print("--- FINAL PREPROCESSING PIPELINE SANITY CHECK ---")
print(f"Training Features Shape (X_train): {X_train_scaled.shape}")
print(f"Testing Features Shape (X_test):   {X_test_scaled.shape}")
print(f"Training Target Distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nScaled Training Means (Should be ~0): {X_train_scaled.mean(axis=0)}")
print(f"Scaled Training StDs  (Should be ~1): {X_train_scaled.std(axis=0)}")

--- FINAL PREPROCESSING PIPELINE SANITY CHECK ---
Training Features Shape (X_train): (252, 2)
Testing Features Shape (X_test):   (109, 2)
Training Target Distribution:
provision_class
Medium Provision    0.50
Low Provision       0.25
High Provision      0.25
Name: proportion, dtype: float64

Scaled Training Means (Should be ~0): [-1.33931666e-16 -1.16309079e-16]
Scaled Training StDs  (Should be ~1): [1. 1.]


### Technical Evaluation: Partitioning Dynamics and Leakage Prevention
* Preservation of Stratified Class Proportions: A 70/30 train-test partition yields a training set size of 252 observations and an independent testing evaluation set of 109 observations. Crucially, the target distribution within the training split perfectly maintains the native population ratio (50% Medium, 25% Low, 25% High). This structural alignment guarantees that downstream optimization handles uniform class boundaries across partitions.

* Mathematical Vector Convergence: Fitting the StandardScaler exclusively to the training split and applying those parameters forward prevents experimental data leakage. The scaled training means converging at $\approx 0$ ($10^{-16}$) with an exact unit variance of $1.0$ confirms the elimination of dimensional scale bias. The structural inputs are formally optimized for multi-attribute technique implementation.

## 3. Technique Implementatio
### 3.1 Baseline Technique Implementation: Linear Support Vector Classifier

In [86]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the baseline Support Vector Classifier with default hyperparameter configurations
baseline_svc = SVC(kernel='linear', random_state=42)

# 2. Train the baseline model using the scaled training partitions
baseline_svc.fit(X_train_scaled, y_train)

# 3. Generate predictions across both partitions to inspect for overfitting
y_pred_train = baseline_svc.predict(X_train_scaled)
y_pred_test = baseline_svc.predict(X_test_scaled)

# 4. Output the complete classification performance metrics for the test partition
print("--- BASELINE SVC TEST PARTITION PERFORMANCE ---")
# Setting zero_division=0 cleans up the local system warnings and keeps the output pristine
print(classification_report(y_test, y_pred_test, zero_division=0))

print("\nRaw Confusion Matrix Indices:")
print(confusion_matrix(y_test, y_pred_test, labels=['Low Provision', 'Medium Provision', 'High Provision']))

--- BASELINE SVC TEST PARTITION PERFORMANCE ---
                  precision    recall  f1-score   support

  High Provision       0.00      0.00      0.00        27
   Low Provision       0.00      0.00      0.00        28
Medium Provision       0.50      1.00      0.66        54

        accuracy                           0.50       109
       macro avg       0.17      0.33      0.22       109
    weighted avg       0.25      0.50      0.33       109


Raw Confusion Matrix Indices:
[[ 0 28  0]
 [ 0 54  0]
 [ 0 27  0]]


### Technical Evaluation: Baseline Performance
* Majority-Class Collapse: The baseline Linear SVC fails to find a meaningful decision boundary, defaulting to guessing Medium Provision for every single local authority. Because the Medium tier accounts for exactly 50% of our test data, the model achieves an illusory 50% global accuracy while completely failing on the High and Low provision tiers (yielding 0.00 for precision and recall).

* Linear Inadequacy: The poor macro-averaged F1-score (0.22) confirms that regional wealth and population dynamics do not share a simple, straight-line relationship with EV infrastructure. Forcing a linear boundary onto this data forces the algorithm to absorb the minority classes into the dominant middle tier, establishing a clear academic justification for moving to a non-linear ensemble approach.

### 3.2 Ensemble Technique Implementation: Random Forest Classifier

In [87]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the Ensemble Random Forest Classifier
# We set max_depth to control overfitting on our smaller regional dataset
ensemble_rf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    random_state=42, 
    class_weight='balanced'
)

# 2. Train the ensemble model using the scaled training features
ensemble_rf.fit(X_train_scaled, y_train)

# 3. Generate predictions across the test partition
y_pred_rf = ensemble_rf.predict(X_test_scaled)

# 4. Output the performance metrics to evaluate the ensemble's capacity
print("--- ENSEMBLE RANDOM FOREST TEST PARTITION PERFORMANCE ---")
# Using zero_division=0 to keep our output completely clean and professional
print(classification_report(y_test, y_pred_rf, zero_division=0))

print("\nRaw Confusion Matrix Indices:")
print(confusion_matrix(y_test, y_pred_rf, labels=['Low Provision', 'Medium Provision', 'High Provision']))

--- ENSEMBLE RANDOM FOREST TEST PARTITION PERFORMANCE ---
                  precision    recall  f1-score   support

  High Provision       0.37      0.41      0.39        27
   Low Provision       0.40      0.61      0.48        28
Medium Provision       0.61      0.41      0.49        54

        accuracy                           0.46       109
       macro avg       0.46      0.47      0.45       109
    weighted avg       0.50      0.46      0.46       109


Raw Confusion Matrix Indices:
[[17  7  4]
 [17 22 15]
 [ 9  7 11]]


### Technical Evaluation: Ensemble Optimization and Class Resolution
* Resolution of Minority Tiers: The Random Forest Classifier successfully breaks the majority-class collapse seen in the baseline. By utilizing an ensemble of non-linear decision trees, the model resolves predictions across all three operational tiers, accurately identifying 17 Low, 22 Medium, and 11 High provision local authorities.

* The Macro-Metric Turnaround: Although global accuracy shifts slightly to 46%, the Macro Average F1-score doubles from 0.22 to 0.45. This metric shift proves that while the problem remains highly complex, moving to a non-parametric ensemble technique allows the pipeline to capture non-linear interactions between local wealth, population scales, and actual charging infrastructure deployment.